# Regularização

O material anterior terminou com um treinamento em que a perda de treino caía até quase zero enquanto a acurácia de validação parava de subir. Nenhuma das técnicas de lá corrige isso, porque todas cuidam do fluxo de sinal dentro da rede, e o problema aqui é outro: o modelo está aprendendo os exemplos em vez do padrão.

Regularizar é restringir o aprendizado para que ele generalize. Este material trata das quatro formas mais usadas, que penalizam pesos grandes, desligam unidades durante o treino, ampliam os dados e interrompem o treinamento no ponto certo.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

In [ ]:
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Overfitting

O sobreajuste, ou overfitting, acontece quando a capacidade do modelo é grande em relação à quantidade de dados. Sobram parâmetros para memorizar particularidades de cada exemplo, inclusive o ruído, e o que foi memorizado não vale para exemplos novos.

Uma forma clássica de organizar o erro de um modelo é

$$
\text{erro} = \text{viés}^2 + \text{variância} + \text{ruído}
$$

em que o viés é o erro por simplificação excessiva, a variância é a sensibilidade do modelo a mudanças nos dados de treino, e o ruído é a parte irredutível do problema. Um modelo grande demais tem viés baixo e variância alta. A regularização troca um pouco de viés por bastante variância a menos.

Para tornar o efeito visível, o treino usa apenas 300 imagens do MNIST, contra 1000 na validação e 1000 no teste.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.1307,), std=(0.3081,)),
])

full_train_set = datasets.MNIST(root="data", train=True, download=True, transform=transform)
full_test_set = datasets.MNIST(root="data", train=False, download=True, transform=transform)

In [ ]:
train_set = Subset(full_train_set, range(300))
validation_set = Subset(full_test_set, range(1000))
test_set = Subset(full_test_set, range(1000, 2000))

train_dataloader = DataLoader(train_set, batch_size=32, shuffle=True)
validation_dataloader = DataLoader(validation_set, batch_size=500, shuffle=False)
test_dataloader = DataLoader(test_set, batch_size=500, shuffle=False)

print(f"treino: {len(train_set)}, validação: {len(validation_set)}, teste: {len(test_set)}")

O modelo tem três camadas ocultas e mais de duzentos mil parâmetros para trezentos exemplos, o que é um exagero deliberado. O argumento `dropout` fica em zero por enquanto e será usado mais adiante.

In [ ]:
class MLP(nn.Module):
    def __init__(self, dropout=0.0):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.layers(x)   # [batch, 10]

In [ ]:
criterion = nn.CrossEntropyLoss()


def evaluate(model, dataloader):
    model.eval()
    total_loss = 0.0
    correct = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            logits = model(images)
            total_loss += criterion(logits, labels).item() * images.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()

    return total_loss / len(dataloader.dataset), correct / len(dataloader.dataset)

In [ ]:
def train(model, dataloader, optimizer, epochs=60):
    history = {"train_loss": [], "validation_loss": [], "validation_accuracy": []}

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            loss = criterion(model(images), labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        validation_loss, validation_accuracy = evaluate(model, validation_dataloader)
        history["train_loss"].append(running_loss / len(dataloader.dataset))
        history["validation_loss"].append(validation_loss)
        history["validation_accuracy"].append(validation_accuracy)

    return history

In [ ]:
def plot_history(histories):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    for name, history in histories.items():
        line = ax1.plot(history["train_loss"], label=f"{name}, treino")[0]
        ax1.plot(history["validation_loss"], linestyle="--", color=line.get_color(),
                 label=f"{name}, validação")
        ax2.plot(history["validation_accuracy"], label=name)

    ax1.set_xlabel("época")
    ax1.set_ylabel("entropia cruzada")
    ax1.legend()
    ax1.grid(True)

    ax2.set_xlabel("época")
    ax2.set_ylabel("acurácia de validação")
    ax2.legend()
    ax2.grid(True)
    plt.show()

In [ ]:
torch.manual_seed(42)
baseline = MLP().to(device)
optimizer = torch.optim.Adam(baseline.parameters(), lr=1e-3)

baseline_history = train(baseline, train_dataloader, optimizer)
print(f"parâmetros: {sum(p.numel() for p in baseline.parameters()):,}")
print(f"perda de treino final: {baseline_history['train_loss'][-1]:.4f}")
print(f"perda de validação final: {baseline_history['validation_loss'][-1]:.4f}")

In [ ]:
plot_history({"sem regularização": baseline_history})

As duas curvas de perda se separam cedo. A de treino desce até perto de zero, porque trezentos exemplos cabem folgadamente na capacidade do modelo, enquanto a de validação atinge um mínimo e volta a subir. Esse é o desenho característico do sobreajuste, e o ponto em que as curvas se separam é o ponto a partir do qual o treinamento passa a piorar o modelo.

A acurácia de validação é menos dramática, porque ela só olha para qual classe teve o maior logit, e não para o quanto o modelo está confiante. A perda captura essa confiança, e é por isso que ela sobe antes.

## Penalização dos pesos

A primeira família de técnicas soma um termo à função de custo,

$$
J = L + \lambda \, \Omega(w)
$$

em que $L$ é a perda original, $\Omega(w)$ mede o tamanho dos pesos e $\lambda$ decide o peso dessa preferência contra o ajuste aos dados. Como o treinamento minimiza a soma, ele passa a preferir soluções com pesos menores, e pesos menores produzem funções mais suaves, menos capazes de acompanhar cada ponto do treino.

As duas escolhas usuais são

$$
\text{L2:} \quad \Omega(w) = \sum_i w_i^2
\qquad
\text{L1:} \quad \Omega(w) = \sum_i |w_i| .
$$

A L2 penaliza mais os pesos grandes e encolhe todos de forma suave, sem zerar nenhum. A L1 aplica a mesma pressão a todos os pesos, independentemente do tamanho, o que empurra os pequenos exatamente para zero e produz modelos esparsos.

In [ ]:
def l1_penalty(model):
    return sum(parameter.abs().sum() for parameter in model.parameters())


def l2_penalty(model):
    return sum((parameter ** 2).sum() for parameter in model.parameters())


print(f"L1 do modelo treinado: {l1_penalty(baseline).item():,.1f}")
print(f"L2 do modelo treinado: {l2_penalty(baseline).item():,.1f}")

Com as penalidades escritas assim, basta somá-las à perda antes do `backward`, e o autograd cuida do resto.

```python
loss = criterion(model(images), labels) + l2_lambda * l2_penalty(model)
```

Para a L2 existe um atalho. Somar $\lambda \sum w_i^2$ ao custo equivale a subtrair uma fração do próprio peso a cada atualização, e todo otimizador do PyTorch implementa isso no argumento `weight_decay`, sem que a perda precise ser alterada.

In [ ]:
model = MLP().to(device)

with_l2 = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
decoupled = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

print(f"weight_decay: {with_l2.param_groups[0]['weight_decay']}")

A equivalência entre somar o termo à perda e usar `weight_decay` vale para o SGD, mas não para o Adam. No Adam o termo entra no gradiente e passa pela normalização adaptativa, de modo que parâmetros com gradientes historicamente grandes acabam sofrendo menos decaimento, que é o contrário do pretendido.

O `AdamW` corrige isso aplicando o decaimento diretamente sobre os pesos, fora do mecanismo adaptativo. Quando a escolha é usar Adam com weight decay, o `AdamW` é a versão a usar.

## Dropout

O dropout ataca o problema por outro lado. A cada passagem para a frente durante o treino, cada unidade da camada é zerada com probabilidade $p$,

$$
\tilde{h} = \frac{h \cdot m}{1 - p}
\qquad
m \sim \text{Bernoulli}(1 - p)
$$

em que $m$ é a máscara sorteada e a divisão por $1 - p$ compensa as unidades removidas, mantendo a média das ativações igual à que se teria sem o dropout. Essa é a versão implementada no PyTorch, e é ela que dispensa qualquer ajuste na hora de avaliar.

O efeito é que nenhuma unidade pode contar com a presença de outra específica, então a rede não consegue construir caminhos frágeis em que um punhado de unidades decide tudo. A informação precisa ficar distribuída.

In [ ]:
dropout = nn.Dropout(p=0.5)
h = torch.ones(2, 8)

dropout.train()
print(f"treino: {dropout(h)[0]}")

dropout.eval()
print(f"avaliação: {dropout(h)[0]}")

No modo de treino, cerca de metade das entradas foi zerada e as demais foram multiplicadas por dois. No modo de avaliação, a camada não faz nada, e a rede inteira é usada. Como no caso da batch normalization, é o `model.train()` e o `model.eval()` que decidem qual dos dois comportamentos vale.

## Data augmentation

As três técnicas anteriores restringem o modelo. Esta age sobre os dados, ampliando o conjunto de treino com versões transformadas dos exemplos que já existem.

Dado o conjunto $D = \{(x_i, y_i)\}$, escolhemos um conjunto de transformações

$$
\mathcal{T} = \{T : \mathcal{X} \to \mathcal{X} \mid y(T(x)) = y(x)\}
$$

isto é, transformações que mudam a entrada sem mudar a classe. A cada época, cada exemplo é apresentado sob uma transformação sorteada, e o modelo nunca vê duas vezes exatamente a mesma imagem.

A condição de preservar a classe é o que decide quais transformações são válidas, e ela depende do problema. Para dígitos, pequenas rotações, translações e mudanças de escala são seguras, porque um 7 continua sendo um 7. Já um espelhamento horizontal não é: ele transforma um 2 em algo que não é dígito nenhum, e pode confundir 6 com algo próximo de um 9 invertido. Espelhar dígitos ensina a rede a aceitar como válido aquilo que não é.

In [ ]:
augmentation = transforms.Compose([
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.1307,), std=(0.3081,)),
])

raw_train_set = datasets.MNIST(root="data", train=True, download=True)
original_image, original_label = raw_train_set[0]

In [ ]:
fig, axes = plt.subplots(1, 7, figsize=(14, 3))

axes[0].imshow(original_image, cmap="gray")
axes[0].set_title(f"original ({original_label})")
axes[0].axis("off")

for ax in axes[1:]:
    ax.imshow(augmentation(original_image).squeeze(), cmap="gray")
    ax.axis("off")
plt.show()

As seis versões são visivelmente diferentes entre si e continuam sendo o mesmo dígito. Como a transformação é aplicada no momento em que o exemplo é lido, nada é armazenado, e a augmentation entra apenas no conjunto de treino: na validação e no teste queremos medir o modelo sobre os dados como eles são.

## Early stopping

O gráfico do começo mostrou que existe uma época a partir da qual o treinamento passa a piorar a validação. O early stopping usa isso diretamente, interrompendo o treino quando a perda de validação deixa de melhorar por um número de épocas seguidas, chamado de paciência.

Como a melhor época quase nunca é a última, guardar o estado do modelo no melhor momento é parte da técnica. O `state_dict` de um módulo é o dicionário com todos os seus tensores, e serve tanto para isso quanto para salvar um modelo em disco.

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5):
        self.patience = patience
        self.best_loss = float("inf")
        self.counter = 0
        self.best_state = None

    def step(self, validation_loss, model):
        if validation_loss < self.best_loss:
            self.best_loss = validation_loss
            self.counter = 0
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1

        return self.counter >= self.patience

In [ ]:
torch.manual_seed(42)
model = MLP().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
early_stopping = EarlyStopping(patience=5)

for epoch in range(60):
    train(model, train_dataloader, optimizer, epochs=1)
    validation_loss, validation_accuracy = evaluate(model, validation_dataloader)

    if early_stopping.step(validation_loss, model):
        print(f"parada na época {epoch + 1}, melhor perda de validação {early_stopping.best_loss:.4f}")
        break

In [ ]:
model.load_state_dict(early_stopping.best_state)
model.to(device)

restored_loss, restored_accuracy = evaluate(model, validation_dataloader)
print(f"modelo restaurado: perda {restored_loss:.4f}, acurácia {restored_accuracy:.4f}")

O treinamento parou bem antes das sessenta épocas, e o modelo restaurado é o da melhor época, não o da última. O custo da técnica é o de sempre: a paciência é mais um hiperparâmetro, e um valor pequeno demais interrompe o treino em uma oscilação passageira.

## Juntando tudo

O modelo abaixo usa dropout de 0.3, `AdamW` com weight decay e dados aumentados, e é comparado com a mesma arquitetura sem nada disso. Os dois treinam pelo mesmo número de épocas, sobre os mesmos 300 exemplos.

In [ ]:
augmented_train_set = Subset(
    datasets.MNIST(root="data", train=True, download=True, transform=augmentation),
    range(300),
)
augmented_dataloader = DataLoader(augmented_train_set, batch_size=32, shuffle=True)

In [ ]:
torch.manual_seed(42)
regularized = MLP(dropout=0.3).to(device)
optimizer = torch.optim.AdamW(regularized.parameters(), lr=1e-3, weight_decay=1e-4)

regularized_history = train(regularized, augmented_dataloader, optimizer)
plot_history({"sem regularização": baseline_history, "com regularização": regularized_history})

In [ ]:
models = {"sem regularização": (baseline, baseline_history), "com regularização": (regularized, regularized_history)}

for name, (model, history) in models.items():
    test_loss, test_accuracy = evaluate(model, test_dataloader)
    print(f"{name}: perda de treino {history['train_loss'][-1]:.4f}, "
          f"acurácia de teste {test_accuracy:.4f}, L2 dos pesos {l2_penalty(model).item():,.1f}")

O modelo regularizado tem perda de treino muito maior, o que é exatamente o esperado: ele foi impedido de memorizar. O que interessa é a perda de validação, que deixa de subir, e a acurácia de teste, que é a medida final e melhora bastante.

Repare que a soma dos quadrados dos pesos não diminuiu, apesar do weight decay. Um valor de 1e-4 é fraco, e o número final resulta das três técnicas agindo ao mesmo tempo, não do decaimento sozinho. Atribuir um efeito a uma técnica específica exige variar uma de cada vez, que é o que os exercícios pedem.

Nenhuma das quatro técnicas é gratuita. Todas introduzem hiperparâmetros, e todas reduzem o ajuste aos dados de treino em troca de uma promessa de generalização que só a validação pode confirmar.

## Exercícios

### Exercício 1

Treine o modelo com dropout em 0.0, 0.3 e 0.6, mantendo o resto igual, e compare as curvas de validação. A partir de que valor a regularização passa a atrapalhar?

In [ ]:
dropouts = [0.0, 0.3, 0.6]

### Exercício 2

Compare a augmentation usada aqui com uma versão exagerada, que inclua rotações de até 90 graus e espelhamento horizontal. O que acontece com a acurácia de teste, e por quê?

In [ ]:
# aug_exagerada = transforms.Compose([...])

### Exercício 3

Aumente o conjunto de treino de 300 para 3000 exemplos e treine o modelo sem nenhuma regularização. Quanto do sobreajuste desaparece só por causa dos dados?

In [ ]:
train_set = Subset(full_train_set, range(3000))